In [59]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

class SimulationDataset(Dataset):
    def __init__(self, root_dir, num_trials=25, use_bulk=False):
        self.root_dir = root_dir
        self.num_trials = num_trials
        self.use_bulk = use_bulk

        # Get all simulation directories (sim1, sim2, ..., simN)
        self.sim_dirs = sorted(
            [d for d in os.listdir(root_dir) if d.startswith("sim")],
            key=lambda x: int(''.join(filter(str.isdigit, x)))
        )

    def __len__(self):
        return len(self.sim_dirs)

    def __getitem__(self, idx):
        sim_path = os.path.join(self.root_dir, self.sim_dirs[idx])
        
        # Load theta
        theta_path = os.path.join(sim_path, "parameters.npy")
        theta_np = np.load(theta_path)[2:]  # Skip first two parameters
        theta = torch.tensor(theta_np, dtype=torch.float32)

        # Load trial results
        trials = []
        for t in range(1, self.num_trials + 1):
            trial_dir = os.path.join(sim_path, str(t))
            file_name = "CNratios_bulk.npy" if self.use_bulk else "CNratios_largest.npy"
            result_path = os.path.join(trial_dir, file_name)

            if os.path.exists(result_path):
                result_np = np.load(result_path)
                trials.append(torch.tensor(result_np, dtype=torch.float32))
            else:
                # Pad missing trial with NaNs of the expected shape
                if trials:
                    nan_tensor = torch.full_like(trials[0], float('nan'))
                else:
                    nan_tensor = torch.zeros(44, dtype=torch.float32).fill_(float('nan'))  # assuming 44 features
                trials.append(nan_tensor)
                #print(f"[WARNING] Missing trial {t} in {sim_path}, padding with NaNs.")

        x_tensor = torch.stack(trials)  # shape: (num_trials, feature_dim)
        return theta, x_tensor

# Load dataset
dataset = SimulationDataset(root_dir="numpy_data", num_trials=25, use_bulk=False)

# Split into train and val sets
total_len = len(dataset)
val_len = int(0.2 * total_len)
train_len = total_len - val_len
train_dataset, val_dataset = random_split(dataset, [train_len, val_len])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [60]:
print(f'train : {len(train_loader)}')
print(f'valid : {len(val_loader)}')

train : 109
valid : 28


In [61]:
for batch_idx, (theta, x) in enumerate(train_loader):
  if batch_idx == 0:
    print(theta.size(), x.size())
    #break

torch.Size([8, 44]) torch.Size([8, 25, 44])


In [62]:
len(val_loader)

28

In [63]:
import importlib
import model  # import the whole module
importlib.reload(model)
from model import InferenceModel
import tqdm as notebook_tqdm


model = InferenceModel(
    train_loader=train_loader,
    val_loader=val_loader,
    hidden_dim_phi=128,
    hidden_dim_rho=128,
    output_dim=44,
    learning_rate=5e-4,
    dropout=0.1,
    max_epochs=200,
    stop_after_epochs=30,
    log_progress=True
)

model.train()

theta device: mps:0
x device: mps:0
torch.Size([8, 44])
torch.Size([8, 25, 44])
[2025-07-25 10:26:47.933364] Epoch 0: train loss: 11.88 | val loss:-2.35
[2025-07-25 10:27:00.555112] Epoch 1: train loss: -0.08 | val loss:-2.29
[2025-07-25 10:27:13.312806] Epoch 2: train loss: -4.86 | val loss:-7.12
[2025-07-25 10:27:27.559572] Epoch 3: train loss: -8.30 | val loss:-8.30
[2025-07-25 10:27:42.144273] Epoch 4: train loss: -9.39 | val loss:-8.62
[2025-07-25 10:27:56.932631] Epoch 5: train loss: -10.81 | val loss:-9.75
[2025-07-25 10:28:11.254803] Epoch 6: train loss: -11.34 | val loss:-10.28
[2025-07-25 10:28:25.264324] Epoch 7: train loss: -12.31 | val loss:-10.88
[2025-07-25 10:28:39.704335] Epoch 8: train loss: -13.51 | val loss:-11.82
[2025-07-25 10:28:53.757701] Epoch 9: train loss: -13.81 | val loss:-11.85
[2025-07-25 10:29:07.459818] Epoch 10: train loss: -14.28 | val loss:-12.63
[2025-07-25 10:29:21.161033] Epoch 11: train loss: -15.11 | val loss:-12.92
[2025-07-25 10:29:34.566489] 

In [64]:
model.save("trained_model.pkl")

### Test model on an unseen Simulated data 

In [65]:
import pickle

with open("trained_model.pkl", "rb") as f:
    model_NF = pickle.load(f)

#model = model.density_estimator.eval()  # Make sure it's in evaluation mode

In [66]:
import torch.nn.functional as F
import torch.nn as nn

model_NF.density_estimator.eval()
mse_list = []
loss_fn = nn.MSELoss()
for theta_true, x_val in val_loader:
    theta_true = theta_true.to(model_NF.device)
    x_val = x_val.to(model_NF.device)

    # Sample from posterior
    samples, _ = model_NF.density_estimator.sample_and_log_prob(
        sample_shape=torch.Size([1000]),
        condition=x_val
    )

    # Take the mean of the posterior samples (mean point estimate)
    theta_pred = samples.mean(dim=0)  # shape: (batch_size, param_dim)

    # Compute MSE for this batch
    
    mse_batch = loss_fn(theta_pred, theta_true)
    #mse_batch = F.mse_loss(theta_pred, theta_true, reduction='mean')
    mse_list.append(mse_batch.item())

In [67]:
mse_list

[0.0351748950779438,
 0.03739124536514282,
 0.038725346326828,
 0.03793115168809891,
 0.036940254271030426,
 0.03894300013780594,
 0.03711479902267456,
 0.037215907126665115,
 0.039548370987176895,
 0.037017736583948135,
 0.039180513471364975,
 0.0390687920153141,
 0.038280315697193146,
 0.04136783629655838,
 0.044096678495407104,
 0.03630939498543739,
 0.036168694496154785,
 0.03820478916168213,
 0.04079702869057655,
 0.04042840376496315,
 0.03571486473083496,
 0.04006939008831978,
 0.03989867493510246,
 0.036055147647857666,
 0.04112013429403305,
 0.03949837014079094,
 0.04109715670347214,
 0.027532145380973816]

## Apply NN model for selection coeeficent parameter estimation

In [68]:
import NN_Utils
import torch
import torch.nn as nn
import torch.optim as optim
importlib.reload(NN_Utils)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model = SimpleRegressor(input_dim=25 * 44, output_dim=44).to(device)
model = NN_Utils.DeepSetClassifier(NN_Utils.embedding_net, embedding_dim=64).to(device)  # Using DeepSetClassifier

optimizer = optim.Adam(model.parameters(), lr=1e-5)
loss_fn = nn.MSELoss()


In [69]:
early_stopping = NN_Utils.EarlyStopping(patience=30, delta=1)

history = {
    "train_loss": [],
    "val_loss": [],
    "train_r2": [],
    "val_r2": [],}

num_epochs = 200
for epoch in range(num_epochs):
    loss_tr, r2_tr = NN_Utils.train(model, train_loader, optimizer, loss_fn, device)
    _, loss_ts, r2_ts = NN_Utils.evaluate(model, val_loader, loss_fn, device)
    '''''
    print(
        f"Epoch {epoch+1:03d} | "
        f"Train: Loss = {loss_tr:.4f}, F1 = {f1_tr:.4f}, Precision = {precision_tr:.4f}, Recall = {recall_tr:.4f} | "
        f"Val: Loss = {loss_ts:.4f}, F1 = {f1_ts:.4f}, Precision = {precision_ts:.4f}, Recall = {recall_ts:.4f}"
    )
    '''''
    print(
        f"Epoch {epoch+1:03d} | "
        f"Train: Loss = {loss_tr:.4f}, R2 Score = {r2_tr:.4f} | "
        f"Val: Loss = {loss_ts:.4f}, R2 Score = {r2_ts:.4f}"
    )
    
    history["train_loss"].append(loss_tr)
    history["val_loss"].append(loss_ts)
    history["train_r2"].append(r2_tr)
    history["val_r2"].append(r2_ts)


    early_stopping(loss_ts, model)
    if early_stopping.early_stop:
        print(f"Early stopping triggered at epoch {epoch+1}.")
        break

# Save model
torch.save(model.state_dict(), "nn_predictor.pth")
print("Model saved to nn_predictor.pth")


Epoch 001 | Train: Loss = 0.0611, R2 Score = -0.6299 | Val: Loss = 0.0537, R2 Score = -0.5190
Epoch 002 | Train: Loss = 0.0526, R2 Score = -0.4620 | Val: Loss = 0.0481, R2 Score = -0.4119
Epoch 003 | Train: Loss = 0.0479, R2 Score = -0.3709 | Val: Loss = 0.0455, R2 Score = -0.3391
Epoch 004 | Train: Loss = 0.0448, R2 Score = -0.3107 | Val: Loss = 0.0441, R2 Score = -0.2853
Epoch 005 | Train: Loss = 0.0444, R2 Score = -0.2669 | Val: Loss = 0.0432, R2 Score = -0.2503
Epoch 006 | Train: Loss = 0.0430, R2 Score = -0.2355 | Val: Loss = 0.0427, R2 Score = -0.2219
Epoch 007 | Train: Loss = 0.0430, R2 Score = -0.2115 | Val: Loss = 0.0422, R2 Score = -0.2014
Epoch 008 | Train: Loss = 0.0419, R2 Score = -0.1918 | Val: Loss = 0.0418, R2 Score = -0.1831
Epoch 009 | Train: Loss = 0.0420, R2 Score = -0.1757 | Val: Loss = 0.0415, R2 Score = -0.1689
Epoch 010 | Train: Loss = 0.0419, R2 Score = -0.1625 | Val: Loss = 0.0412, R2 Score = -0.1573
Epoch 011 | Train: Loss = 0.0414, R2 Score = -0.1521 | Val: 

In [53]:
importlib.reload(NN_Utils)
loss_list, avg_loss, avg_r2 = NN_Utils.evaluate(model, val_loader, loss_fn, device)

In [54]:
loss_list

[0.03141281381249428,
 0.03256640210747719,
 0.03981820121407509,
 0.03601202741265297,
 0.03630516678094864,
 0.03284871205687523,
 0.038900185376405716,
 0.038888100534677505,
 0.03447166457772255,
 0.03428133949637413,
 0.036083925515413284,
 0.03231584280729294,
 0.03713707998394966,
 0.03696814179420471,
 0.036536701023578644,
 0.036551352590322495,
 0.03831373155117035,
 0.03669727221131325,
 0.04172585904598236,
 0.03545016422867775,
 0.03197096288204193,
 0.03856445476412773,
 0.036920882761478424,
 0.03841445595026016,
 0.031842295080423355,
 0.03394164890050888,
 0.03628121688961983,
 0.02820192649960518]

In [57]:
np.array(mse_list)>np.array(loss_list)

array([ True,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True,  True,  True, False,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False])